In [ ]:
import pandas as pd
from pathlib import Path

base_path = Path("/mnt/volume6/czj/labLGN/LabLZ/data_preparation/")
file_path = base_path / "mmc1.xlsx"

In [1]:
import re

aa3to1 = {
    'Ala': 'A', 'Arg': 'R', 'Asn': 'N', 'Asp': 'D', 'Cys': 'C',
    'Gln': 'Q', 'Glu': 'E', 'Gly': 'G', 'His': 'H', 'Ile': 'I',
    'Leu': 'L', 'Lys': 'K', 'Met': 'M', 'Phe': 'F', 'Pro': 'P',
    'Ser': 'S', 'Thr': 'T', 'Trp': 'W', 'Tyr': 'Y', 'Val': 'V'
}


# Input Format Example: LDHA K222E
def extract_mutation(variant_str, gene):
    if not isinstance(variant_str, str):
        return None
    
    # Remove the gene prefix if present
    prefix = str(gene) + " "
    mut = variant_str[len(prefix):] if variant_str.startswith(prefix) else variant_str.strip()
    
    # K222E
    if re.match(r'^[A-Z]\d+[A-Z]$', mut):
        return mut
    
    # Tyr418H
    m = re.match(r'^([A-Z][a-z]{2})(\d+)([A-Z])$', mut)
    if m:
        aa3, pos, new_aa = m.group(1), m.group(2), m.group(3)
        if aa3 in aa3to1:
            return f"{aa3to1[aa3]}{pos}{new_aa}"
    
    return None

def make_key(df, variant_col):
    # use gene and mutation to create a unique key for each row
    mut = df.apply(lambda r: extract_mutation(r[variant_col], r["Gene"]), axis=1)
    return df["Gene"].astype(str) + " " + mut.astype(str)

# Sheet "Variant annotation"

In [4]:
df = pd.read_excel(file_path, sheet_name="Variant annotation")
print(f"Number of rows: {len(df)}")
print(f"Column names: {df.columns.tolist()}")

Number of rows: 3448
Column names: ['Gene', 'Variant', 'Variant (alternative)', 'Collection', 'Phenotype annotation', 'dbSNP_ID', 'Uniprot', 'Mislocalized', 'Mislocalization phenotype', 'Impact score', 'AlphaMissense score', 'AlphaMissense class', 'GnomAD VEP frequency', 'Inheritance', 'ClinVar class', 'PPI phenotype', 'PTM residue', 'PTM type', 'Disulfide bond', 'HeLa expression (TPM)']


In [5]:
df_clean = df[df["Mislocalized"].isin([0, 1])].copy()
print(f"\nAfter filtering NAs: {len(df_clean)} rows")
print(df_clean["Mislocalized"].value_counts())


After filtering NAs: 2280 rows
Mislocalized
0.0    2030
1.0     250
Name: count, dtype: int64


In [6]:
cols = ["Gene", "Variant", "Variant (alternative)", "Uniprot",
        "Mislocalized", "Mislocalization phenotype",
        "AlphaMissense score", "AlphaMissense class",
        "ClinVar class", "HeLa expression (TPM)"]
df_model = df_clean[cols].copy()
df_model["source"] = "main"
print(f"\ndf_model: {len(df_model)} rows, UniProt missing: {df_model['Uniprot'].isna().sum()}")



df_model: 2280 rows, UniProt missing: 111


# Sheet "Additional benign variants"

In [10]:
df_benign = pd.read_excel(file_path, sheet_name="Additional benign variants")

# filter NAs 
df_benign_clean = df_benign[df_benign["Mislocalized?"].isin([0, 1])].copy()
print(f"\nAdditional benign: {len(df_benign_clean)} rows")
print(df_benign_clean["Mislocalized?"].value_counts())

# check for overlap between the two datasets
va_keys  = set(make_key(df_model, "Variant"))
abv_keys = set(make_key(df_benign_clean, "Variant"))
overlap  = va_keys & abv_keys
print(f"Overlap rows: {len(overlap)}")  # is 0

# align column formats
df_benign_aligned = pd.DataFrame({
    "Gene":                      df_benign_clean["Gene"].values,
    "Variant":                   df_benign_clean["Variant"].values,
    "Variant (alternative)":     None,
    "Uniprot":                   None,
    "Mislocalized":              df_benign_clean["Mislocalized?"].astype(int).values,
    "Mislocalization phenotype": None,
    "AlphaMissense score":       None,
    "AlphaMissense class":       None,
    "ClinVar class":             None,
    "HeLa expression (TPM)":     None,
    "source":                    "additional_benign"
})

df_combined = pd.concat([df_model, df_benign_aligned], ignore_index=True)
print(f"\nTotal rows after combining: {len(df_combined)}")
print(df_combined["Mislocalized"].value_counts())


Additional benign: 95 rows
Mislocalized?
0.0    94
1.0     1
Name: count, dtype: int64
Overlap rows: 0

Total rows after combining: 2375
Mislocalized
0.0    2124
1.0     251
Name: count, dtype: int64


/tmp/ipykernel_4086015/3542041535.py:29: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined = pd.concat([df_model, df_benign_aligned], ignore_index=True)


In [12]:
df_combined["Mutation"] = df_combined.apply(
    lambda row: extract_mutation(row["Variant"], row["Gene"]), axis=1)

df_combined["Mutation (alternative)"] = df_combined.apply(
    lambda row: extract_mutation(row["Variant (alternative)"], row["Gene"]), axis=1)

print(f"Mutation extraction successful: {df_combined['Mutation'].notna().sum()}/{len(df_combined)}")
print(f"Mutation (alternative) extraction successful: {df_combined['Mutation (alternative)'].notna().sum()}/{len(df_combined)}")

Mutation extraction successful: 2338/2375
Mutation (alternative) extraction successful: 234/2375


# Sheet "Localization screen results"

In [13]:
df_screen = pd.read_excel(file_path, sheet_name="Localization screen results")

# Mutation == "reference"
step1 = df_screen[df_screen["Mutation"] == "reference"]

# Drop duplicates based on the "Gene" column
step2 = step1.drop_duplicates(subset="Gene")

# Only keep the "Gene" and "Primary location" columns
step3 = step2[["Gene", "Primary location"]]

# Rename the "Primary location" column to "wt_primary" 
df_wt = step3.rename(columns={"Primary location": "wt_primary"})

df_combined = df_combined.merge(df_wt, on="Gene", how="left")
print(f"\n{df_combined['wt_primary'].isna().sum()} rows have missing wt_primary values")


2 rows have missing wt_primary values


In [14]:
df_combined.to_csv(base_path / "cell2024_combined.csv", index=False)
print(f"Saved: {len(df_combined)} rows and {len(df_combined.columns)} columns")

Saved: 2375 rows and 14 columns
